# **章节实践**
通过本章的系统学习，我们掌握了如何分析模型中需要优化的算子以及将优化后的算子应用到模型的全部流程。为了巩固所学知识，现提供以下实践练习：

基于Qwen2.5-0.5B-Instruct模型，优化其中至少一处Mul算子的调用

要求：

1.通过Ascend Pytorch Profiler工具找到一处Mul算子的调用位置以及对应的算子场景。  

2.开发自定义Mul算子MulCustom并完成部署  

3.开发MulCustom算子插件

4.替换原Mul算子并通过Ascend Pytorch Profiler工具确认MulCustom成功调用

5.模型输出精度正常

请开始你的实践，体验完整开发过程。

---

## **环境准备：**

In [ ]:
!mkdir -p Sources/04

import os, subprocess
env = subprocess.check_output("bash -l -c 'source $ASCEND_TOOLKIT_HOME/set_env.sh && env'", shell=True, text=True)
for line in env.splitlines():
    if "=" in line: os.environ.__setitem__(*line.split("=", 1))
print("\n🎉 Environment initialization process completed successfully!")

安装相关依赖：

In [ ]:
!pip3 install modelscope expecttest transformers==5.3.0 -i https://pypi.tuna.tsinghua.edu.cn/simple -t Sources/04
import os 
import sys
pkg_dir = os.path.abspath("Sources/04")
if pkg_dir not in sys.path:
    sys.path.insert(0, pkg_dir)

下载模型：

In [ ]:
!modelscope download --model Qwen/Qwen2.5-0.5B-Instruct  --local_dir Sources/04/Qwen 

---
## **模型推理脚本：**
可以通过以下脚本推理模型进行验证

In [ ]:
import transformers
from transformers import AutoModelForCausalLM, AutoTokenizer

model_name = "Sources/04/Qwen/"

model = AutoModelForCausalLM.from_pretrained(
    model_name
).half().npu()
tokenizer = AutoTokenizer.from_pretrained(model_name)

prompt = "Give me a short introduction to large language model."
messages = [
    {"role": "system", "content": "You are Qwen, created by Alibaba Cloud. You are a helpful assistant."},
    {"role": "user", "content": prompt}
]
text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)
model_inputs = tokenizer([text], return_tensors="pt").to(model.device)

generated_ids = model.generate(
    **model_inputs,
    max_new_tokens=512
)
generated_ids = [
    output_ids[len(input_ids):] for input_ids, output_ids in zip(model_inputs.input_ids, generated_ids)
]

response = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]
